# JTAGENT fast QLoRA (T4-safe)

Run cells in order. This bounded run uses 2,000 rows, one epoch, 512-token sequences, and 250 optimizer steps. It archives the finished adapter to Drive before optional export work.

In [ ]:
# 1. Fresh T4 runtime setup.
import subprocess, sys
from pathlib import Path
def run(*args): subprocess.run(args, check=True)
run('nvidia-smi', '-L')
if not Path('/content/jayti/.git').is_dir(): run('git', 'clone', '--depth', '1', 'https://github.com/jaytipargal/jayti', '/content/jayti')
run(sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44', 'peft>=0.12', 'datasets>=2.20', 'accelerate>=0.33', 'bitsandbytes>=0.43', 'huggingface_hub', 'sentencepiece', 'protobuf')


In [ ]:
# 2. Mount Drive and obtain the gated-model token from Colab Secrets.
import os
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)
assert Path('/content/drive/MyDrive').is_dir(), 'Drive mount failed.'
token = userdata.get('HF_TOKEN')
if not token: raise RuntimeError('Add an HF_TOKEN secret with gated-read access, then rerun this cell.')
os.environ['HF_TOKEN'] = token


In [ ]:
# 3. Fetch, redact, and reduce the training set deterministically.
import glob, json, random, shutil
from huggingface_hub import snapshot_download
sys.path[:0] = ['/content/jayti/sandbox/jtagent', '/content/jayti/scripts']
import segment_jsonl
ROOT = Path('/content/TAN/jtagent'); ROOT.mkdir(parents=True, exist_ok=True)
snap = snapshot_download('jtagent/jt-agent-data', repo_type='dataset', allow_patterns=['training/daily_ingestion/**'])
items = []
for source in glob.glob(f'{snap}/training/daily_ingestion/**/*.jsonl', recursive=True):
    for line in open(source, encoding='utf-8'):
        try: items.append(json.loads(line))
        except json.JSONDecodeError: pass
chunks = [c for c in (segment_jsonl._item_to_chunk(item) for item in items) if c]
segment_jsonl.write_category_jsonl(ROOT, segment_jsonl.redact_chunks(chunks))
train = ROOT / 'training'; backup = ROOT / 'training_backup_full'
if not backup.exists(): shutil.copytree(train, backup)
random.seed(42)
for file in (train / 'extracted_text').glob('*.jsonl'):
    rows = [line for line in file.read_text(errors='replace').splitlines() if line.strip()]
    if len(rows) > 2000: rows = [rows[i] for i in sorted(random.sample(range(len(rows)), 2000))]
    file.write_text('\n'.join(rows) + '\n', encoding='utf-8')
for file in (train / 'browser_data').glob('*.jsonl'): file.write_text('', encoding='utf-8')


In [ ]:
# 4. Stop only an old copy of this driver, then start one detached run.
import os, signal, time
for process in Path('/proc').glob('[0-9]*'):
    try:
        if 'run_t4_fast.py' in (process / 'cmdline').read_bytes().decode(errors='replace'):
            os.kill(int(process.name), signal.SIGTERM)
    except (FileNotFoundError, ProcessLookupError, PermissionError): pass
time.sleep(3)
driver = ROOT / 'run_t4_fast.py'
driver_code = r'''import json, os, shutil, sys, traceback
from datetime import date, datetime, timezone
from pathlib import Path
ROOT = Path('/content/TAN/jtagent')
STATUS = ROOT / 'AGENT_RUN_STATUS.json'
PERSIST = Path('/content/drive/MyDrive/JTAGENT_runs')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path.insert(0, '/content/jayti/sandbox/jtagent')
def status(stage, **extra):
    STATUS.write_text(json.dumps({'stage': stage, 'updated_at': datetime.now(timezone.utc).isoformat(), **extra}, indent=2) + '\n')
try:
    status('training')
    import qlora_train
    cfg = qlora_train.build_config({'epochs': 1, 'max_steps': 250, 'save_steps': 125, 'save_total_limit': 2, 'batch_size': 1, 'grad_accum': 8, 'max_seq_len': 512})
    result = qlora_train.train_qlora(ROOT, config=cfg)
    adapter = result.get('adapter')
    if not adapter: raise RuntimeError('QLoRA did not produce an adapter: {}'.format(result))
    if not Path('/content/drive/MyDrive').is_dir(): raise RuntimeError('Google Drive is not mounted')
    PERSIST.mkdir(parents=True, exist_ok=True)
    archive = shutil.make_archive(str(PERSIST / 'adapter_{}_qlora'.format(date.today().isoformat())), 'gztar', root_dir=adapter)
    status('completed', adapter=adapter, adapter_archive=archive)
    print('DRIVER_DONE', flush=True)
except Exception as exc:
    status('failed', error_type=type(exc).__name__, error=str(exc))
    traceback.print_exc()
    print('DRIVER_ERROR', flush=True)
    raise
'''
driver.write_text(driver_code, encoding='utf-8')
log = ROOT / 'agent_train_export.log'
with log.open('a') as out:
    proc = subprocess.Popen([sys.executable, '-u', str(driver)], stdout=out, stderr=subprocess.STDOUT, start_new_session=True)
print({'driver_pid': proc.pid, 'log': str(log)})


In [ ]:
# 5. Read-only status; rerun this cell as needed.
import json
status_file = ROOT / 'AGENT_RUN_STATUS.json'; log = ROOT / 'agent_train_export.log'
print(json.loads(status_file.read_text()) if status_file.exists() else 'No status file yet.')
print(log.read_text(errors='replace')[-4000:] if log.exists() else 'No log yet.')
